In [2]:
import numpy as np
import gurobipy as gp
from gurobipy import GRB

In [6]:
n_samples = 100 
n_locations = 2
demands = np.random.rand(n_samples, n_locations)

In [20]:
noises = np.random.rand(223, demands[0,:].shape[0])
(demands[0,:] + noises).shape


(223, 2)

In [14]:
def generate_samples(n_samples, model, X, rho): 
    means = model(X) 
    noises = np.random.rand(n_samples, means.shape[0])
    return means + noises


In [13]:
def saa(demands, holding, backorder, edges): 
    K = demands.shape[0]
    N = demands.shape[1] 

    solver = gp.Model('nominal_solver')

    a_ = [(k,i,j) for k in range(K) for i in range(N) for j in range(N)]
    q_ = [i for i in range(N)]
    a = solver.addVars(a_, lb = 0, name='a')
    q = solver.addVars(q_, lb = 0, name='q')

    for k in range(K): 
        for i in range(N): 
            solver.addConstr(sum(a[k,i,j] for j in range(N)) <= q[i])
    
    solver.setObjective(
            sum(holding *   sum(q[i] - sum(a[k,i,j] for j in range(N)) for i in range(N)) for k in range(K) +
                backorder * sum(demands[k,i] - sum(a[k,j,i] for j in range(N)) for i in range(N)) for k in range(K) +
                sum(edges[i,j] * a[k,i,j] for i in range(N) for j in range(N))
            ), GRB.MINIMIZE)
        
    solver.optimize()
    return solver

In [12]:
saa(demands)

Gurobi Optimizer version 9.5.1 build v9.5.1rc2 (win64)
Thread count: 8 physical cores, 16 logical processors, using up to 16 threads
Optimize a model with 0 rows, 402 columns and 0 nonzeros
Model fingerprint: 0x39f63d9c
Coefficient statistics:
  Matrix range     [0e+00, 0e+00]
  Objective range  [0e+00, 0e+00]
  Bounds range     [0e+00, 0e+00]
  RHS range        [0e+00, 0e+00]
Presolve removed 0 rows and 402 columns
Presolve time: 0.01s
Presolve: All rows and columns removed
Iteration    Objective       Primal Inf.    Dual Inf.      Time
       0    0.0000000e+00   0.000000e+00   0.000000e+00      0s

Solved in 0 iterations and 0.01 seconds (0.00 work units)
Optimal objective  0.000000000e+00


<gurobi.Model Continuous instance nominal_solver: 0 constrs, 402 vars, No parameter changes>